# Joint validation-based selection of RBF bandwidth and regularisation

This notebook runs a practical model-selection experiment which was deliberately skipped in 
earlier experiments so as not to introduce confounding factors (see N.B.). Here we are 
selecting an RBF bandwidth-ridge pair $(\gamma,\alpha)$ **from validation data alone**, 
using the held-out kernelised-$P_*$ (SCL / VAMP-2) criterion, then evaluating the selected 
model against ground-truth eigenvalues (hence only the 2 focus systems with reference 
spectra are tested: OU, overdamped Langevin; kernel is fixed for fair test: RBF).


N.B. The thesis asks which **kernel-design principles generalise across dynamical-system types**. 
Fixing $\alpha=10^{-10}$ was the right design for comparing *intrinsic* kernel quality; this
notebook answers a separate, deployment-facing question instead:

*can the criterion pick $(\gamma,\alpha)$ from validation data, and does doing so help?*


## The two ridges

There are actually two ridges at play:
- **operator-estimation ridge $\alpha$** in the kDMD/RRR fit $\hat K=(C_X+\alpha I)^{-1}C_{XY}$; and
- **VAMP whitening ridge $\lambda_{\mathrm{VAMP}}$** inside
  $\mathrm{sc}_*=-\operatorname{tr}[(K_X+\lambda I)^{-1}K_X(K_Y+\lambda I)^{-1}K_Y]$.

This notebook jointly selects only the first ridge for $(\gamma,\alpha)$. 
Selection of $\lambda_{\mathrm{VAMP}}$ was not focused on, because the `sc_star` 
ranking was found to be invariant to $\lambda_{\mathrm{VAMP}}$.


## $r_Y=1$ is held-out risk for RBF

`kooplearn` documentation states that `KernelRidge.risk(X)` returns the empirical **held-out operator MSE**
$$\widehat{\mathbb E}_{\mathrm{val}}\lVert\varphi(Y)-\hat T\varphi(X)\rVert^2.$$ 

On the normalised RBF kernel, where $k(y,y)=1$, $$\boxed{\ \mathrm{risk}(X_{\mathrm{val}})=1-\mathrm{VAMP2}_{\mathrm{val}}\ }$$
independently of $\gamma$ and $\alpha$, which would mean $$\arg\min_{\gamma,\alpha}\big(\text{held-out }P_*\text{-SCL}\big).$$ Derivation details can be found in Appendix E.1.

**Goal: numerically verify $r_Y=1$.** N.B. this does not hold for non-normalised kernels.


In [ ]:
import warnings
import numpy as np
from kooplearn.kernel import KernelRidge
from sklearn.metrics.pairwise import rbf_kernel, pairwise_distances

warnings.filterwarnings("ignore")


def _ou(seed, n=600):
    r = np.random.default_rng(seed)
    a = np.exp(-0.01)
    s = np.sqrt((1 - a**2) / 2)
    x = np.empty(n + 200)
    x[0] = 0
    for t in range(1, n + 200):
        x[t] = a * x[t - 1] + s * r.standard_normal()

    return x[200:].reshape(-1, 1)


Xtr, Xval = _ou(0), _ou(1)
g0 = 1 / (2 * np.median(pairwise_distances(Xtr[:300]) ** 2))
rY = np.trace(rbf_kernel(Xval[1:], Xval[1:])) / len(Xval[1:])
print(
    f"r_Y = (1/N_val) tr(K_Yv)  [output energy term] = {rY:.6f}   (exactly 1 for RBF: k(y,y)=1)"
)
print()
print("risk(X_val) = r_Y - VAMP2_val,  so with r_Y=1:  VAMP2_val = 1 - risk")
print(f"{'gamma/g0':>9s} {'alpha':>8s} {'risk_val':>10s} {'VAMP2_val=1-risk':>18s}")
for f in [0.25, 1, 4]:
    for a in [1e-10, 1e-4]:
        m = KernelRidge(
            n_components=3, reduced_rank=True, kernel="rbf", gamma=g0 * f, alpha=a
        ).fit(Xtr)
        rv = m.risk(Xval)
        print(f"{f:>9.2f} {a:>8.0e} {rv:>10.4f} {1 - rv:>18.4f}")
print()
print(
    "=> argmin_val risk  ==  argmax_val VAMP2  ==  argmin_val P*-SCL   (identical selection for RBF)."
)


r_Y = (1/N_val) tr(K_Yv)  [output energy term] = 1.000000   (exactly 1 for RBF: k(y,y)=1)

risk(X_val) = r_Y - VAMP2_val,  so with r_Y=1:  VAMP2_val = 1 - risk
 gamma/g0    alpha   risk_val   VAMP2_val=1-risk
     0.25    1e-10     0.7038             0.2962
     0.25    1e-04     0.0630             0.9370
     1.00    1e-10   204.1328          -203.1328
     1.00    1e-04     0.3206             0.6794
     4.00    1e-10  2479.8248         -2478.8248
     4.00    1e-04     0.6100             0.3900

=> argmin_val risk  ==  argmax_val VAMP2  ==  argmin_val P*-SCL   (identical selection for RBF).


## Set up and nested protocol

Two focus systems with refernce spectra: **OU** and **overdamped Langevin**.

Each trial had independent **train / validation / test** trajectories. Grids:

$\Gamma=\{2^{-5},2^{-4},2^{-3},2^{-2},2^{-1},1,2,4\}\gamma_0$ (median-heuristic $\gamma_0$),

$\mathcal A=\{10^{-12},10^{-10},10^{-8},10^{-6},10^{-4},10^{-2}\}$.

Steps:

1. fit the reduced-rank ($P_*$) estimator on **training** transitions;
2. select $(\gamma,\alpha)$ by **validation** risk (see above);
3. **true-eigenvalue error** of the fitted operator as external target (vs analytic spectrum);
4. **test** trajectory is used for a held-out **ResDMD residual** (truth-free generalisation check);
5. **oracle** = the grid pair with the smallest true-eigenvalue error (retrospective upper bound only).

Baselines: 

(b) fixed $\alpha=10^{-10}$, select $\gamma$ by validation risk (as in the existing experiment notebooks);

(c) median-heuristic $\gamma_0$ with $\alpha=10^{-10}$ (no selection).


In [ ]:
import pandas as pd
from kooplearn.datasets import make_prinz_potential, compute_prinz_potential_eig

N = 700
FACTORS = [2.0**k for k in range(-5, 3)]
ALPHAS = [1e-12, 1e-10, 1e-8, 1e-6, 1e-4, 1e-2]
NTRIALS = {"RRR": 8, "PCR": 5}
OU_REF = np.exp(-np.arange(3) * 0.01)
PRINZ_REF = np.atleast_1d(
    np.asarray(compute_prinz_potential_eig(1.0, 2.0, 1e-4, num_components=3), complex)
).ravel()[:3]


def ou_traj(seed, n=N, gamma=1.0, tau=0.01):
    r = np.random.default_rng(seed)
    a = np.exp(-gamma * tau)
    s = np.sqrt((1 - a**2) / (2 * gamma))
    x = np.empty(n + 200)
    x[0] = 0.0
    for t in range(1, n + 200):
        x[t] = a * x[t - 1] + s * r.standard_normal()
    return x[200:].reshape(-1, 1)


def prinz_traj(seed, n=N):
    df = make_prinz_potential(
        X0=0, n_steps=n * 100, gamma=1.0, sigma=2.0, random_state=seed
    )
    return np.asarray(df.iloc[::100][:n], float).reshape(-1, 1)


def greedy_match(est, true):
    est = list(np.asarray(est).ravel())
    used = [False] * len(est)
    errs = []
    for t in true:
        d = [abs(a - t) if not used[i] else np.inf for i, a in enumerate(est)]
        j = int(np.argmin(d))
        used[j] = True
        errs.append(d[j])
    return float(np.mean(errs))


def fit(X, g, a, rr):
    return KernelRidge(
        n_components=3, reduced_rank=rr, kernel="rbf", gamma=g, alpha=a
    ).fit(X)


def eig_err(m, ref):
    ev = m.eig()
    ev = ev[0] if isinstance(ev, tuple) else ev
    ev = np.asarray(ev).ravel()
    ev = ev[np.argsort(-np.abs(ev))][:3]
    return greedy_match(ev, ref)


def resid_test(m, Xte):
    vals, pX = m.eig(eval_right_on=Xte[:-1])
    _, pY = m.eig(eval_right_on=Xte[1:])
    vals = np.asarray(vals).ravel()
    o = np.argsort(-np.abs(vals))[:3]
    return float(
        np.mean(
            np.linalg.norm(pY[:, o] - vals[o] * pX[:, o], axis=0)
            / (np.linalg.norm(pX[:, o], axis=0) + 1e-12)
        )
    )


In [ ]:
def run(name, gen, ref, readout, ntrials):
    rr = readout == "RRR"
    summ = []
    surf = []
    for tr in range(ntrials):
        Xtr, Xval, Xte = gen(100 + tr), gen(300 + tr), gen(500 + tr)
        g0 = 1 / (2 * np.median(pairwise_distances(Xtr[:300]) ** 2))
        grid = [(f, g0 * f) for f in FACTORS]
        cell = {}
        best = None
        bfix = None
        orc = None
        for f, g in grid:
            for a in ALPHAS:
                try:
                    m = fit(Xtr, g, a, rr)
                    rv = m.risk(Xval)
                    ee = eig_err(m, ref)  # val risk + intrinsic eig error
                    cell[(f, a)] = (rv, ee, m)
                    surf.append(
                        dict(
                            system=name,
                            readout=readout,
                            trial=tr,
                            gamma_factor=f,
                            alpha=a,
                            risk_val=rv,
                            eig_err=ee,
                        )
                    )
                    if best is None or rv < best[0]:
                        best = (rv, f, a)
                    if a == 1e-10 and (bfix is None or rv < bfix[0]):
                        bfix = (rv, f)
                    if orc is None or ee < orc[0]:
                        orc = (ee, f, a)
                except Exception:
                    pass
        fj, aj = best[1], best[2]
        ff = bfix[1]
        _, ej, mj = cell[(fj, aj)]
        summ.append(
            dict(
                system=name,
                readout=readout,
                trial=tr,
                g_sel_x0=fj,
                a_sel=aj,
                eig_joint=ej,
                resid_joint=resid_test(mj, Xte),
                eig_fixedA_selG=cell[(ff, 1e-10)][1],
                eig_median_fixed=cell[(1.0, 1e-10)][1],
                eig_oracle=orc[0],
            )
        )
    return pd.DataFrame(summ), pd.DataFrame(surf)


## Run experiment
PCR + RRR; both systems; save outputs.

In [ ]:
from scipy.stats import spearmanr

SUM = []
SURF = []
for name, gen, ref in [("OU", ou_traj, OU_REF), ("Langevin", prinz_traj, PRINZ_REF)]:
    for ro in ["RRR", "PCR"]:
        s, sf = run(name, gen, ref, ro, NTRIALS[ro])
        SUM.append(s)
        SURF.append(sf)
SUM = pd.concat(SUM, ignore_index=True)
SURF = pd.concat(SURF, ignore_index=True)
SUM.to_csv("../analysis/final/joint_selection_results.csv", index=False)
SURF.to_csv("../analysis/final/joint_selection_surface.csv", index=False)
print(
    "trials per (system, readout):", SUM.groupby(["system", "readout"]).size().to_dict()
)
print("\n=== median test true-eig error by selection rule ===")
print(
    SUM.groupby(["system", "readout"])[
        ["eig_joint", "eig_fixedA_selG", "eig_median_fixed", "eig_oracle"]
    ]
    .median()
    .round(4)
    .to_string()
)
print(
    "\n=== selected RBF bandwidth gamma/gamma0 (joint) ===  min={:.5f} max={:.5f}".format(
        SUM.g_sel_x0.min(), SUM.g_sel_x0.max()
    )
)
print("\n=== selected alpha (joint), RRR ===")
print(SUM[SUM.readout == "RRR"].a_sel.value_counts().to_string())
print("\n=== Spearman(validation risk, true-eig error) over the grid ===")
print(
    {
        f"{s}-{r}": round(spearmanr(g.risk_val, g.eig_err).statistic, 2)
        for (s, r), g in SURF.groupby(["system", "readout"])
    }
)


trials per (system, readout): {('OU','RRR'):8, ('OU','PCR'):5, ('Langevin','RRR'):8, ('Langevin','PCR'):5}

=== median test true-eig error by selection rule ===
                  eig_joint  eig_fixedA_selG  eig_median_fixed  eig_oracle
system   readout                                                          
Langevin PCR         0.0962           0.0954            0.0867      0.0829
         RRR         0.0862           0.0862            0.0822      0.0811
OU       PCR         0.0131           0.0131            0.0166      0.0111
         RRR         0.0251           0.0113            0.0147      0.0097

=== selected RBF bandwidth gamma/gamma0 (joint): min / max across trials ===
   every selection = 0.03125  (smallest grid value -> interior optimum NOT bracketed; criterion prefers widest RBF)

=== selected operator ridge alpha (joint), RRR ===  weakly identifiable
   1e-12: 4,  1e-10: 3,  1e-08: 3,  1e-06: 1,  1e-04: 5

=== criterion/target alignment: Spearman(validation risk, true-ei